# EasyRAG Generation failures and guardrails

## Chapter position

A decent retrieval result is not the end of the story. Generation still needs rules for what to do when evidence is weak, when citations are required, or when the answer model itself fails.

## Learning question

Which branches inside `synthesize_answer()` control abstention, normal answer-model use, and fallback behavior?

## Success criteria

- You can map unsupported evidence to the early-abstain branch.
- You can show the difference between a normal answer-model path and a fallback path.
- You can explain how strict and loose answer settings change the returned metadata.

## Source code anchors

- `easyrag.rag.generation.synthesis.synthesize_answer`
- `easyrag.rag.generation.pipeline.generate_answer`
- `easyrag.rag.evaluation.grounding.answer_abstained`
- `easyrag.rag.types.AnswerParam`

## Method principles

- `easyrag.rag.generation.synthesis.synthesize_answer`: This method owns the final synthesis branch logic. It decides whether to abstain, call the answer model, or fall back to heuristic grounded synthesis, then enforces citation policy.
- `easyrag.rag.generation.pipeline.generate_answer`: This is the answer orchestration wrapper. It reruns citation selection, packs context, builds the prompt, calls synthesis, and returns a structured `AnswerResult`.
- `easyrag.rag.evaluation.grounding.answer_abstained`: This heuristic checks whether the answer explicitly abstained. It matters because a grounded system should sometimes refuse rather than overclaim.
- `easyrag.rag.types.AnswerParam`: This dataclass is the answer-side policy bundle. It controls citation budget, context budget, answer style, and abstention rules without changing retrieval behavior.

Design reason: these anchors break answering into evidence selection, packing, prompting, and synthesis so the notebook can show that answer quality is a data-shaping problem as much as a model-wording problem.


## How the code fits together

`synthesize_answer()` has three practical branches in this chapter. If the citations do not support the question and `allow_abstain` is on, it returns an abstention early. If an answer model is available and works, the function uses that model and then enforces citation markers if needed. If the answer model raises an exception, the function falls back to heuristic grounded synthesis instead of crashing the whole request. `generate_answer()` wraps those branches and exposes the result through `AnswerResult.metadata`.

Design reason: the notebook walks the same evidence through smaller answer-stage boundaries before it asks for a final sentence. That ordering is what lets you see whether a failure came from evidence budget, context packing, prompting policy, or synthesis.


## What this cell is proving

We will use one deterministic workspace and then hold the retrieval result steady while we change only the generation settings and answer-model behavior.

In [ ]:
import importlib
import sys
import tempfile
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "notebooks" / "_utils.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

_notebook_utils = importlib.import_module("notebooks._utils")
_rag = importlib.import_module("easyrag.rag")
_grounding = importlib.import_module("easyrag.rag.evaluation.grounding")
_pipeline = importlib.import_module("easyrag.rag.generation.pipeline")
_async_utils = importlib.import_module("easyrag.support.async_utils")

ensure_repo_root_on_path = _notebook_utils.ensure_repo_root_on_path
_print_json = _notebook_utils.print_json
can_use_openai_compatible_models = _notebook_utils.provider_ready
skip_message = _notebook_utils.skip_message
_stub_embedding = _notebook_utils.stub_embedding
_stub_query_model = _notebook_utils.stub_query_model
AnswerParam = _rag.AnswerParam
EasyRAG = _rag.EasyRAG
QueryParam = _rag.QueryParam
answer_abstained = _grounding.answer_abstained
answer_has_citations = _grounding.answer_has_citations
generate_answer = _pipeline.generate_answer
run_sync = _async_utils.run_sync

REPO_ROOT = ensure_repo_root_on_path()

## What this cell is proving

The supported and unsupported questions should hit the same workspace. That way any behavior change you see later really belongs to generation, not to a different corpus.

Design reason: generation is intentionally split into evidence selection, packing, prompting, and synthesis because each boundary can fail for a different reason. Keeping this step explicit is what makes answer quality debuggable instead of treating the model response as one indivisible artifact.


In [ ]:
guard_tmp = tempfile.TemporaryDirectory()
guard_rag = EasyRAG(
    working_dir=guard_tmp.name,
    workspace="guardrails-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(guard_rag.initialize_storages())
run_sync(
    guard_rag.ainsert(
        [
            "# Retrieval\nEasyRAG uses grounded retrieval and query rewrite.\n",
            "# Policy\nCitation-aware answers stay inspectable.\n",
        ],
        ids=["doc::retrieval", "doc::policy"],
        file_paths=["docs/retrieval.md", "docs/policy.md"],
    )
)

supported_question = "How does EasyRAG use query rewrite?"
unsupported_question = "When is the company retreat?"
supported_result = run_sync(
    guard_rag.aquery(
        supported_question,
        QueryParam(
            mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3
        ),
    )
)
unsupported_result = run_sync(
    guard_rag.aquery(
        unsupported_question,
        QueryParam(
            mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3
        ),
    )
)

strict_param = AnswerParam(
    require_citations=True, allow_abstain=True, max_citations=2, max_context_chars=150
)
loose_param = AnswerParam(
    require_citations=False, allow_abstain=False, max_citations=1, max_context_chars=80
)

_print_json(
    {
        "supported_citation_locations": [
            citation["location"] for citation in supported_result.citations
        ],
        "unsupported_citation_locations": [
            citation["location"] for citation in unsupported_result.citations
        ],
    }
)

## Why this output looks like this

The unsupported question can still retrieve citations because lexical overlap and fallback retrieval do not know the question is unsupported in a business sense. The abstention decision happens later, when generation checks whether those citations actually support the question.

Design reason: the output looks layered because the same evidence is being carried through several answer-stage representations. Keeping the citation subset, packed context, prompt text, and final answer close together is what exposes whether a failure came from selection, packing, instruction design, or synthesis.


## What this cell is proving

With strict settings and weak evidence, generation should stop early and abstain. This is the "insufficient evidence" branch inside `synthesize_answer()`.

Design reason: generation is intentionally split into evidence selection, packing, prompting, and synthesis because each boundary can fail for a different reason. Keeping this step explicit is what makes answer quality debuggable instead of treating the model response as one indivisible artifact.


In [ ]:
strict_unsupported = generate_answer(
    unsupported_question,
    unsupported_result,
    answer_param=strict_param,
    answer_model_func=None,
)

_print_json(
    {
        "answer": strict_unsupported.answer,
        "metadata": strict_unsupported.metadata,
        "abstained": answer_abstained(strict_unsupported.answer),
        "has_citations": answer_has_citations(strict_unsupported.answer),
    }
)

## Why this output looks like this

`allow_abstain=True` lets the function return early once the citations fail the support check. In that branch, `fallback_used` is still `true` because the answer came from the built-in guardrail path rather than a model call, and `insufficient_evidence` is set so you can tell why it happened.

Design reason: the output looks layered because the same evidence is being carried through several answer-stage representations. Keeping the citation subset, packed context, prompt text, and final answer close together is what exposes whether a failure came from selection, packing, instruction design, or synthesis.


## What this cell is proving

When a working answer model is available, synthesis uses that model first. Citation enforcement still happens after the model returns.

Design reason: generation is intentionally split into evidence selection, packing, prompting, and synthesis because each boundary can fail for a different reason. Keeping this step explicit is what makes answer quality debuggable instead of treating the model response as one indivisible artifact.


In [ ]:
def grounded_answer_model(prompt: str, **kwargs) -> str:
    return "EasyRAG uses query rewrite before retrieval so the retriever starts from a clearer query."


strict_supported = generate_answer(
    supported_question,
    supported_result,
    answer_param=strict_param,
    answer_model_func=grounded_answer_model,
)

_print_json(
    {
        "answer": strict_supported.answer,
        "metadata": strict_supported.metadata,
        "abstained": answer_abstained(strict_supported.answer),
        "has_citations": answer_has_citations(strict_supported.answer),
    }
)

## Why this output looks like this

The custom answer model returns plain text, then EasyRAG adds citation scaffolding because `require_citations=True`. That is why `answer_model_used` is `true` while `fallback_used` is `false`.

Design reason: the output looks layered because the same evidence is being carried through several answer-stage representations. Keeping the citation subset, packed context, prompt text, and final answer close together is what exposes whether a failure came from selection, packing, instruction design, or synthesis.


## What this cell is proving

If the answer model throws, generation falls back to heuristic grounded synthesis instead of failing the whole request. Loose settings also change how strict the final output needs to be.

Design reason: generation is intentionally split into evidence selection, packing, prompting, and synthesis because each boundary can fail for a different reason. Keeping this step explicit is what makes answer quality debuggable instead of treating the model response as one indivisible artifact.


In [ ]:
def failing_answer_model(prompt: str, **kwargs):
    raise RuntimeError("answer model unavailable")


failing_supported = generate_answer(
    supported_question,
    supported_result,
    answer_param=strict_param,
    answer_model_func=failing_answer_model,
)
loose_supported = generate_answer(
    supported_question,
    supported_result,
    answer_param=loose_param,
    answer_model_func=None,
)

_print_json(
    {
        "failing_supported": {
            "answer": failing_supported.answer,
            "metadata": failing_supported.metadata,
            "abstained": answer_abstained(failing_supported.answer),
            "has_citations": answer_has_citations(failing_supported.answer),
        },
        "loose_supported": {
            "answer": loose_supported.answer,
            "metadata": loose_supported.metadata,
            "abstained": answer_abstained(loose_supported.answer),
            "has_citations": answer_has_citations(loose_supported.answer),
        },
    }
)

## Why this output looks like this

The failing-model case still returns a grounded answer because the exception path drops into the fallback synthesizer. The loose case changes two things at once: it does not allow abstention, and it does not require citation markers. That is why the answer text can be shorter and less visibly grounded even though the retrieved evidence was the same.

Design reason: the output looks layered because the same evidence is being carried through several answer-stage representations. Keeping the citation subset, packed context, prompt text, and final answer close together is what exposes whether a failure came from selection, packing, instruction design, or synthesis.


## What this cell is proving

The provider-backed path should still report the same guardrail metadata even when the answer text itself is produced by a configured model stack.

In [ ]:
if not can_use_openai_compatible_models():
    print(skip_message("provider"))
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(
        working_dir=provider_tmp.name, workspace="provider-guardrails-demo"
    )
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(
            provider_rag.ainsert(
                "# Retrieval\nGrounded retrieval keeps answers inspectable.\n",
                ids=["doc::retrieval"],
                file_paths=["docs/retrieval.md"],
            )
        )
        provider_answer = run_sync(
            provider_rag.aanswer(
                "How do answers stay inspectable?",
                QueryParam(mode="hybrid", chunk_top_k=2),
                AnswerParam(),
            )
        )
        _print_json(
            {
                "answer": provider_answer.answer,
                "metadata": {
                    key: provider_answer.metadata[key]
                    for key in (
                        "answer_model_used",
                        "fallback_used",
                        "raw_citation_count",
                        "selected_citation_count",
                    )
                },
            }
        )
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()

## Common mistakes / debugging cues

- Unsupported retrieval is not the same thing as empty retrieval. You can still get citations and still need to abstain.
- `answer_model_used=True` does not guarantee the final text came straight from the model. Check `fallback_used` too.
- If strict and loose settings produce similar answers, compare `require_citations` and `allow_abstain` before you assume the guardrails are broken.

In [ ]:
run_sync(guard_rag.finalize_storages())
guard_tmp.cleanup()
print("Cleaned up the generation-guardrails workspace.")

## Takeaway

Guardrails mostly change how the system fails. The important questions are whether it abstains when evidence is weak, whether it falls back cleanly when the answer model breaks, and whether the metadata tells you which branch you actually hit.

From here, jump to [../06_evaluation/06_04_answer_grounding_and_faithfulness.ipynb](../06_evaluation/06_04_answer_grounding_and_faithfulness.ipynb) if you want to score these choices systematically.